In [0]:
dbutils.library.restartPython()

In [0]:
from common_utils.data_quailtychecks import dataquailty

In [0]:
import json
from pyspark.sql.functions import col, sum
from pyspark.sql.functions import trim
from common_utils.data_quailtychecks import dataquailty
from common_utils.logging import get_logger
logger = get_logger("Data Engineering")
# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path","")
config_path=dbutils.widgets.get("path")
print(config_path)
# =========================================
# 1. Read CONFIG File
# =========================================
with open(config_path,mode='r') as f:
    config=json.load(f)

# =========================================
# 1. Get Data
# =========================================
source_path=config['source']['source_path']
shrinkage_type_id=config['columns']["shrinkage_type_id"]
shrinkage_type=config['columns']["shrinkage_type"]
category=config['columns']["category"]
is_paid=config['columns']["is_paid"]
is_planned=config['columns']["is_planned"]
print(source_path)
distinct_columns = [
    shrinkage_type,
    category,
    is_paid,
    is_planned
]

# =========================================
# 1. Read Source File
# =========================================
df=spark.read.table(source_path)
df.display()

# =========================================
# 1. Data Quailty Checks
# =========================================
dataquailty(df,shrinkage_type_id,distinct_columns)


# =========================================
# 5. Writing into Silver Layer
# =========================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS group_project.silver"
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config['target']['target_table'])
)

logger.info(
    "Successfully loaded into Silver",

)
